In [1]:
import pyarrow
import pandas as pd
import numpy as np
import holidays
import calendar

In [3]:
df_consumption = pd.read_parquet('../data/hackathonlicence/consumption.parquet')
df_consumption['date'] = pd.to_datetime(df_consumption['date'], format='%Y-%m')
df_consumption.head()

,date,territory_id,category,value
0,2023-01-01,1,Продовольствие,7692
1,2023-01-01,1,Здоровье,1271
2,2023-01-01,1,Маркетплейсы,2505
3,2023-01-01,1,Общественное питание,1142
5,2023-01-01,1,Транспорт,1718


In [4]:
df_market_access = pd.read_parquet('../data/hackathonlicence/market_access.parquet')
df_market_access.head()

,territory_id,market_access
0,1,309.6
1,2,322.8
2,3,315.2
3,4,315.5
4,5,320.2


In [5]:
df_dict_districts = pd.read_excel('../data/t_dict_municipal/t_dict_municipal_districts.xlsx')
df_dict_districts['oktmo_8'] = df_dict_districts['oktmo'].str.replace('-', '').str[:8]
df_dict_districts.head()

,municipal_district_name_short,oktmo,municipal_district_name,municipal_district_type,municipal_district_status,shape,shape_linked_oktmo,municipal_district_center,source_rosstat,year_from,year_to,territory_id,change_id_from,change_id_to,region_code,region_name,municipal_district_center_lat,municipal_district_center_lon,oktmo_8
0,Майкоп,79-701-000-000,городской округ город Майкоп,городской округ,административный_центр_субъекта,1,NaN,г Майкоп,data-20180110-structure-20150128.csv,2018,9999,1,NaN,NaN,1,Республика Адыгея,44.606208,40.104053,79701000
1,Адыгейск,79-703-000-000,городской округ город Адыгейск,городской округ,NaN,1,NaN,г Адыгейск,data-20180110-structure-20150128.csv,2018,9999,2,NaN,NaN,1,Республика Адыгея,44.883378,39.190962,79703000
2,Гиагинский,79-605-000-000,Гиагинский муниципальный район,муниципальный район,NaN,1,NaN,ст-ца Гиагинская,data-20180110-structure-20150128.csv,2018,9999,3,NaN,NaN,1,Республика Адыгея,44.875585,40.056498,79605000
3,Кошехабльский,79-615-000-000,Кошехабльский муниципальный район,муниципальный район,NaN,1,NaN,аул Кошехабль,data-20180110-structure-20150128.csv,2018,9999,4,NaN,NaN,1,Республика Адыгея,44.896500,40.496363,79615000
4,Красногвардейский,79-618-000-000,Красногвардейский муниципальный район,муниципальный район,NaN,1,NaN,с Красногвардейское,data-20180110-structure-20150128.csv,2018,9999,5,NaN,NaN,1,Республика Адыгея,45.144356,39.586148,79618000


In [7]:
df_key_rate_and_inflation = pd.read_excel('../data/Инфляция и ключевая ставка Банка России_F17_09_2013_T15_09_2026.xlsx', dtype={'Дата': str}).iloc[::-1].reset_index(drop=True)
df_key_rate_and_inflation.rename(columns={'Дата': 'date', 'Ключевая ставка, % годовых': 'key_rate', 'Инфляция, % г/г': 'inflation'}, inplace=True)
df_key_rate_and_inflation['date'] = pd.to_datetime(df_key_rate_and_inflation['date'], format='%m.%Y')
df_key_rate_and_inflation.head()

/Users/wicdt/Library/jupyterlab-desktop/jlab_server/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,date,key_rate,inflation,Цель по инфляции
0,2013-09-01,5.5,6.14,NaN
1,2013-10-01,5.5,6.27,NaN
2,2013-11-01,5.5,6.50,NaN
3,2013-12-01,5.5,6.47,NaN
4,2014-01-01,5.5,6.07,NaN


In [8]:
df_consumer_spending = pd.read_parquet('../data/consumer-spending_ru_1788769445730.parquet')
df_consumer_spending['period'] = pd.to_datetime(df_consumer_spending['period'], format='%Y-%m-%d').dt.strftime('%Y-%m')
df_consumer_spending.rename(columns={'period': 'date', 'value': 'national_spending'}, inplace=True)
df_consumer_spending['date'] = pd.to_datetime(df_consumer_spending['date'], format='%Y-%m')
df_consumer_spending.head()

,date,national_spending,obs_status,source,type,ref_area,freq,decimals,unit_measure,unit_mult
0,2018-12-01,4372.54,A,Данные СберИндекса,Всего,Россия,Месяц,2,млрд. руб.,0
1,2019-01-01,3470.37,A,Данные СберИндекса,Всего,Россия,Месяц,2,млрд. руб.,0
2,2019-02-01,3411.93,A,Данные СберИндекса,Всего,Россия,Месяц,2,млрд. руб.,0
3,2019-03-01,3689.41,A,Данные СберИндекса,Всего,Россия,Месяц,2,млрд. руб.,0
4,2019-04-01,3664.45,A,Данные СберИндекса,Всего,Россия,Месяц,2,млрд. руб.,0


In [9]:
df_currency = pd.read_excel('../data/RC_F01_01_2023_T30_09_2026.xlsx')
df_currency.rename(columns={'data': 'date', 'curs': 'usd_rub'}, inplace=True)
df_currency['date'] = pd.to_datetime(df_currency['date'])
df_currency['month_start'] = df_currency['date'].dt.to_period('M').dt.to_timestamp()
df_currency_avg_monthly = df_currency.groupby('month_start')['usd_rub'].mean().reset_index()
df_currency_avg_monthly.rename(columns={'month_start': 'date'}, inplace=True)

df_currency_avg_monthly.head()

/Users/wicdt/Library/jupyterlab-desktop/jlab_server/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,date,usd_rub
0,2023-01-01,68.875975
1,2023-02-01,72.782750
2,2023-03-01,76.103586
3,2023-04-01,80.995543
4,2023-05-01,79.212595


In [10]:
df_population_size = pd.read_parquet('../data/БДМО/data_Y48112027_112_v20250918/data_Y48112027_112_v20250918.parquet')
df_population_size.head()

,indicator_section_code,indicator_section,indicator_code,indicator_name,mest,region_id,region_name,mun_level,mun_district,municipality,...,mun_type_oktmo,oktmo_stable,oktmo_history,oktmo_year_from,oktmo_year_to,year,indicator_value,indicator_unit,indicator_period,comment
0,31,Население,Y48112027,Оценка численности населения на 1 января текущ...,Все население,01,Алтайский край,Муниципальное образование верхнего уровня,Залесовский муниципальный округ,Залесовский муниципальный округ,...,Муниципальный округ,01512000,Изменен тип и название,2022,2025,2022,12979.0,Человек,На 1 января,CD
1,31,Население,Y48112027,Оценка численности населения на 1 января текущ...,Городское население,01,Алтайский край,Муниципальное образование верхнего уровня,Залесовский муниципальный округ,Залесовский муниципальный округ,...,Муниципальный округ,01512000,Изменен тип и название,2022,2025,2022,NaN,Человек,На 1 января,CD
2,31,Население,Y48112027,Оценка численности населения на 1 января текущ...,Сельское население,01,Алтайский край,Муниципальное образование верхнего уровня,Залесовский муниципальный округ,Залесовский муниципальный округ,...,Муниципальный округ,01512000,Изменен тип и название,2022,2025,2022,12979.0,Человек,На 1 января,CD
3,31,Население,Y48112027,Оценка численности населения на 1 января текущ...,Сельское население,01,Алтайский край,Муниципальное образование верхнего уровня,Залесовский муниципальный округ,Залесовский муниципальный округ,...,Муниципальный округ,01512000,Изменен тип и название,2022,2025,2023,11444.0,Человек,На 1 января,CD
4,31,Население,Y48112027,Оценка численности населения на 1 января текущ...,Городское население,01,Алтайский край,Муниципальное образование верхнего уровня,Залесовский муниципальный округ,Залесовский муниципальный округ,...,Муниципальный округ,01512000,Изменен тип и название,2022,2025,2023,NaN,Человек,На 1 января,CD


In [11]:
df_avg_month_salary = pd.read_parquet('../data/БДМО/data_Y48123007_112_v20250918/data_Y48123007_112_v20250918.parquet')
df_avg_month_salary.head()

,indicator_section_code,indicator_section,indicator_code,indicator_name,okved,region_id,region_name,mun_level,mun_district,municipality,...,mun_type_oktmo,oktmo_stable,oktmo_history,oktmo_year_from,oktmo_year_to,year,indicator_value,indicator_unit,indicator_period,comment
0,32,Занятость и заработная плата,Y48123007,Среднемесячная заработная плата работников орг...,Раздел I. Транспорт и связь,01,Алтайский край,Муниципальное образование верхнего уровня,Алейский муниципальный район,Алейский муниципальный район,...,Муниципальный район,01601000,Без изменений,2010,2025,2008,8938.6,Рубль,Январь-декабрь,CD
1,32,Занятость и заработная плата,Y48123007,Среднемесячная заработная плата работников орг...,Раздел J. Финансовая деятельность,01,Алтайский край,Муниципальное образование верхнего уровня,Алейский муниципальный район,Алейский муниципальный район,...,Муниципальный район,01601000,Без изменений,2010,2025,2008,13519.2,Рубль,Январь-декабрь,CD
2,32,Занятость и заработная плата,Y48123007,Среднемесячная заработная плата работников орг...,Раздел L. Государственное управление и обеспеч...,01,Алтайский край,Муниципальное образование верхнего уровня,Алейский муниципальный район,Алейский муниципальный район,...,Муниципальный район,01601000,Без изменений,2010,2025,2008,10178.2,Рубль,Январь-декабрь,CD
3,32,Занятость и заработная плата,Y48123007,Среднемесячная заработная плата работников орг...,Раздел D. Обрабатывающие производства,01,Алтайский край,Муниципальное образование верхнего уровня,Алейский муниципальный район,Алейский муниципальный район,...,Муниципальный район,01601000,Без изменений,2010,2025,2008,4568.3,Рубль,Январь-декабрь,CD
4,32,Занятость и заработная плата,Y48123007,Среднемесячная заработная плата работников орг...,Раздел Е Производство и распределение электроэ...,01,Алтайский край,Муниципальное образование верхнего уровня,Алейский муниципальный район,Алейский муниципальный район,...,Муниципальный район,01601000,Без изменений,2010,2025,2008,5363.8,Рубль,Январь-декабрь,CD


In [12]:
df_avg_month_salary['year'].unique()

<IntegerArray>
[2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016]
Length: 9, dtype: Int64

# нормальный вывод и в ограничения что не беру зп

In [13]:
df_avg_org_headcount = pd.read_parquet('../data/БДМО/data_Y48423005_112_v20250918/data_Y48423005_112_v20250918.parquet')
df_avg_org_headcount.head()

,indicator_section_code,indicator_section,indicator_code,indicator_name,okved2,region_id,region_name,mun_level,mun_district,municipality,...,mun_type_oktmo,oktmo_stable,oktmo_history,oktmo_year_from,oktmo_year_to,year,indicator_value,indicator_unit,indicator_period,comment
0,32,Занятость и заработная плата,Y48423005,Среднесписочная численность работников организ...,Всего по обследуемым видам экономической деяте...,01,Алтайский край,Муниципальное образование верхнего уровня,Залесовский муниципальный округ,Залесовский муниципальный округ,...,Муниципальный округ,01512000,Изменен тип и название,2022,2025,2022,1313.0,Человек,Январь-сентябрь,CD
1,32,Занятость и заработная плата,Y48423005,Среднесписочная численность работников организ...,Раздел O Государственное управление и обеспече...,01,Алтайский край,Муниципальное образование верхнего уровня,Залесовский муниципальный округ,Залесовский муниципальный округ,...,Муниципальный округ,01512000,Изменен тип и название,2022,2025,2022,261.0,Человек,Январь-март,CD
2,32,Занятость и заработная плата,Y48423005,Среднесписочная численность работников организ...,"Раздел D Обеспечение электрической энергией, г...",01,Алтайский край,Муниципальное образование верхнего уровня,Залесовский муниципальный округ,Залесовский муниципальный округ,...,Муниципальный округ,01512000,Изменен тип и название,2022,2025,2022,42.0,Человек,Январь-декабрь,CD
3,32,Занятость и заработная плата,Y48423005,Среднесписочная численность работников организ...,"Раздел G Торговля оптовая и розничная, ремонт ...",01,Алтайский край,Муниципальное образование верхнего уровня,Залесовский муниципальный округ,Залесовский муниципальный округ,...,Муниципальный округ,01512000,Изменен тип и название,2022,2025,2022,61.0,Человек,Январь-декабрь,CD
4,32,Занятость и заработная плата,Y48423005,Среднесписочная численность работников организ...,Раздел Н Транспортировка и хранение,01,Алтайский край,Муниципальное образование верхнего уровня,Залесовский муниципальный округ,Залесовский муниципальный округ,...,Муниципальный округ,01512000,Изменен тип и название,2022,2025,2022,84.0,Человек,Январь-сентябрь,CD


In [14]:
df_avg_org_headcount.columns

Index(['indicator_section_code', 'indicator_section', 'indicator_code',
       'indicator_name', 'okved2', 'region_id', 'region_name', 'mun_level',
       'mun_district', 'municipality', 'oktmo', 'mun_type', 'mun_type_oktmo',
       'oktmo_stable', 'oktmo_history', 'oktmo_year_from', 'oktmo_year_to',
       'year', 'indicator_value', 'indicator_unit', 'indicator_period',
       'comment'],
      dtype='object')

## Подготовка данных БДМО

### Проблема дублирующихся записей

При первичном мерже датафрейм разросся до **110 млн строк** вместо ожидаемых ~500 тысяч.
Причина — дубликаты в файлах БДМО по ключу `oktmo_stable + year`.

В ходе отладки выявлены три источника дублей:

**1. Мусорные значения в `oktmo_stable`**
Некоторые строки содержали `"CD"` вместо ОКТМО.
→ Решение: фильтр `oktmo_stable.str.len() == 8`

**2. Выделенные территории**
Новый округ Сириус (ОКТМО `03731000`) был выделен из Сочи в 2021 году,
но унаследовал `oktmo_stable = 03726000` — как у Сочи.
Это создавало два МО с одинаковым стабильным ключом.
→ Решение: фильтр `oktmo == oktmo_stable` — берём только оригинальную территорию (тут это условие не применяется, что то с max() там)

**3. Полные дубли и ошибки в данных Росстата**
- Новокузнецкий: строка за 2022 год задублирована с идентичным значением
- Теучежский район: за 2011 год два значения (5740 и 20651) — одно явно ошибочное
  (ряд: 20641 → 20584 → **5740**/20651 → 20883)
→ Решение: `groupby(['oktmo_8', 'year'])['population'].max()` — убирает полные дубли
  и автоматически выбирает правдоподобное значение при конфликте

In [40]:
pop_clean = df_population_size[
    df_population_size['mest'] == 'Все население'][['oktmo_stable', 'year', 'indicator_value']].rename(
    columns={'indicator_value': 'population', 'oktmo_stable': 'oktmo_8'}
)
head_clean = df_avg_org_headcount[
    df_avg_org_headcount['okved2'] == 'Всего по обследуемым видам экономической деятельности'][['oktmo_stable', 'year', 'indicator_value']].rename(
    columns={'indicator_value': 'avg_headcount', 'oktmo_stable': 'oktmo_8'}
)
print(df_consumption.shape)
print(pop_clean.groupby(['oktmo_8', 'year']).size().max())
print(head_clean.groupby(['oktmo_8', 'year']).size().max())

(303126, 4)
15428
114


In [16]:
dupes_pop = pop_clean[pop_clean.duplicated(['oktmo_8', 'year'], keep=False)]
print(dupes_pop.head(20)) 
dupes_head = head_clean[head_clean.duplicated(['oktmo_8', 'year'], keep=False)]
print(dupes_head.head(20))

    oktmo_8  year  population
94       CD  2013      1091.0
98       CD  2014      1006.0
99       CD  2015      1099.0
104      CD  2016      1106.0
107      CD  2017      1106.0
109      CD  2018      1084.0
112      CD  2019      1028.0
114      CD  2020       960.0
119      CD  2021       890.0
122      CD  2022       769.0
124      CD  2023      1294.0
127      CD  2024      1244.0
130      CD  2025      1160.0
132      CD  2013       348.0
136      CD  2013       598.0
138      CD  2014       607.0
141      CD  2015       600.0
146      CD  2016       592.0
149      CD  2017       595.0
150      CD  2018       587.0
      oktmo_8  year  avg_headcount
0    01512000  2022         1313.0
8    01512000  2022         1327.0
19   01512000  2022         1354.0
25   01512000  2022         1317.0
37   01512000  2023         1320.0
47   01512000  2023         1280.0
54   01512000  2023         1276.0
55   01512000  2023         1299.0
69   01512000  2024         1251.0
74   01512000  2024 

In [17]:
print(df_population_size['indicator_period'].unique())
print(df_avg_org_headcount['indicator_period'].unique())

<StringArray>
['На 1 января', 'Значение показателя за год']
Length: 2, dtype: string
<StringArray>
['Январь-сентябрь', 'Январь-март', 'Январь-декабрь', 'Январь-июнь']
Length: 4, dtype: string


In [18]:
pop_clean = df_population_size[
    (df_population_size['mest'] == 'Все население') &
    (df_population_size['indicator_period'] == 'На 1 января') &
    (df_population_size['oktmo_stable'].str.len() == 8)
][['oktmo_stable', 'year', 'indicator_value', 'mun_level']].rename(
    columns={'indicator_value': 'population', 'oktmo_stable': 'oktmo_8'}
)

dupes = pop_clean[pop_clean.duplicated(['oktmo_8', 'year'], keep=False)]
print(dupes.head(10))
print(dupes['mun_level'].unique())

head_clean = df_avg_org_headcount[
    (df_avg_org_headcount['okved2'] == 'Всего по обследуемым видам экономической деятельности') &
    (df_avg_org_headcount['indicator_period'] == 'Январь-декабрь')
][['oktmo_stable', 'year', 'indicator_value', 'mun_level']].rename(
    columns={'indicator_value': 'avg_headcount', 'oktmo_stable': 'oktmo_8'}
)

dupes = head_clean[head_clean.duplicated(['oktmo_8', 'year'], keep=False)]
print(dupes.head(10))
print(dupes['mun_level'].unique())

        oktmo_8  year  population                                  mun_level
7131   01616000  2009     12307.0  Муниципальное образование верхнего уровня
7134   01616000  2010     12159.0  Муниципальное образование верхнего уровня
7138   01616000  2011     11970.0  Муниципальное образование верхнего уровня
7141   01616000  2012     11719.0  Муниципальное образование верхнего уровня
7144   01616000  2013     11433.0  Муниципальное образование верхнего уровня
7146   01616000  2014     11127.0  Муниципальное образование верхнего уровня
7149   01616000  2015     10730.0  Муниципальное образование верхнего уровня
18692  01539000  2009     10362.0  Муниципальное образование верхнего уровня
18694  01539000  2010     10293.0  Муниципальное образование верхнего уровня
18696  01539000  2011     10172.0  Муниципальное образование верхнего уровня
<StringArray>
['Муниципальное образование верхнего уровня']
Length: 1, dtype: string
         oktmo_8  year  avg_headcount  \
51497   03726000  2022     

In [19]:
dupes = pop_clean[pop_clean.duplicated(['oktmo_8', 'year'], keep=False)]
print(df_population_size[
    df_population_size['oktmo_stable'] == '01616000'
][['municipality', 'mun_district', 'oktmo', 'oktmo_stable', 'year', 'indicator_value', 'mest']].head(20).to_string())

                       municipality                   mun_district     oktmo oktmo_stable  year  indicator_value                 mest
7131  Каменский муниципальный район  Каменский муниципальный район  01616000     01616000  2009          12307.0        Все население
7132  Каменский муниципальный район  Каменский муниципальный район  01616000     01616000  2009              NaN  Городское население
7133  Каменский муниципальный район  Каменский муниципальный район  01616000     01616000  2009          12307.0   Сельское население
7134  Каменский муниципальный район  Каменский муниципальный район  01616000     01616000  2010          12159.0        Все население
7135  Каменский муниципальный район  Каменский муниципальный район  01616000     01616000  2010              NaN  Городское население
7136  Каменский муниципальный район  Каменский муниципальный район  01616000     01616000  2010          12159.0   Сельское население
7137  Каменский муниципальный район  Каменский муниципальный р

In [20]:
pop_clean = df_population_size[
    (df_population_size['mest'] == 'Все население') &
    (df_population_size['indicator_period'] == 'На 1 января') &
    (df_population_size['oktmo_stable'].str.len() == 8) &
    (df_population_size['mun_district'] == df_population_size['municipality'])
][['oktmo_stable', 'year', 'indicator_value']].rename(
    columns={'indicator_value': 'population', 'oktmo_stable': 'oktmo_8'}
)
print(pop_clean.groupby(['oktmo_8', 'year']).size().max())

3


In [21]:
dupes_h = head_clean[head_clean.duplicated(['oktmo_8', 'year'], keep=False)]
print(df_avg_org_headcount[
    df_avg_org_headcount['oktmo_stable'] == dupes_h['oktmo_8'].iloc[0]
][['municipality', 'mun_district', 'oktmo', 'oktmo_stable', 'year', 'indicator_value', 'indicator_period', 'mun_level']].head(20).to_string())

            municipality       mun_district     oktmo oktmo_stable  year  indicator_value indicator_period                                  mun_level
51104  Город-курорт Сочи  Город-курорт Сочи  03726000     03726000  2017          11160.0   Январь-декабрь  Муниципальное образование верхнего уровня
51105  Город-курорт Сочи  Город-курорт Сочи  03726000     03726000  2017           2419.0   Январь-декабрь  Муниципальное образование верхнего уровня
51106  Город-курорт Сочи  Город-курорт Сочи  03726000     03726000  2017            806.0      Январь-март  Муниципальное образование верхнего уровня
51107  Город-курорт Сочи  Город-курорт Сочи  03726000     03726000  2017            703.0      Январь-март  Муниципальное образование верхнего уровня
51108  Город-курорт Сочи  Город-курорт Сочи  03726000     03726000  2017            702.0  Январь-сентябрь  Муниципальное образование верхнего уровня
51109  Город-курорт Сочи  Город-курорт Сочи  03726000     03726000  2017          23449.0      Январ

In [22]:
dupes_h = head_clean[head_clean.duplicated(['oktmo_8', 'year'], keep=False)]
print(df_avg_org_headcount[
    (df_avg_org_headcount['oktmo_stable'] == '03726000') &
    (df_avg_org_headcount['indicator_period'] == 'Январь-декабрь') &
    (df_avg_org_headcount['mun_level'] == 'Муниципальное образование верхнего уровня') &
    (df_avg_org_headcount['okved2'] == 'Всего по обследуемым видам экономической деятельности')
].to_string())

      indicator_section_code             indicator_section indicator_code                                                                                 indicator_name                                                 okved2 region_id         region_name                                  mun_level       mun_district       municipality     oktmo                                                     mun_type                                               mun_type_oktmo oktmo_stable oktmo_history oktmo_year_from oktmo_year_to  year  indicator_value indicator_unit indicator_period comment
51134                     32  Занятость и заработная плата      Y48423005  Среднесписочная численность работников организаций (без субъектов малого предпринимательства)  Всего по обследуемым видам экономической деятельности        03  Краснодарский край  Муниципальное образование верхнего уровня  Город-курорт Сочи  Город-курорт Сочи  03726000  Городской округ, городской округ с внутригородским делением  Городс

In [23]:
pop_clean = df_population_size[
    (df_population_size['mest'] == 'Все население') &
    (df_population_size['indicator_period'] == 'На 1 января') &
    (df_population_size['oktmo_stable'].str.len() == 8) &
    (df_population_size['oktmo'] == df_population_size['oktmo_stable'])
][['oktmo_stable', 'year', 'indicator_value']].rename(
    columns={'indicator_value': 'population', 'oktmo_stable': 'oktmo_8'}
)

head_clean = df_avg_org_headcount[
    (df_avg_org_headcount['okved2'] == 'Всего по обследуемым видам экономической деятельности') &
    (df_avg_org_headcount['indicator_period'] == 'Январь-декабрь') &
    (df_avg_org_headcount['mun_level'] == 'Муниципальное образование верхнего уровня') &
    (df_avg_org_headcount['oktmo'] == df_avg_org_headcount['oktmo_stable'])  # ← то же самое
][['oktmo_stable', 'year', 'indicator_value']].rename(
    columns={'indicator_value': 'avg_headcount', 'oktmo_stable': 'oktmo_8'}
)

print(pop_clean.groupby(['oktmo_8', 'year']).size().max())
print(head_clean.groupby(['oktmo_8', 'year']).size().max())

2
1


In [24]:
dupes = pop_clean[pop_clean.duplicated(['oktmo_8', 'year'], keep=False)]
print(dupes.head(20).to_string())

         oktmo_8  year  population
207895  32731000  2022    536388.0
207896  32731000  2022    536388.0
444766  79633000  2011      5740.0
444767  79633000  2011     20651.0


In [25]:
problem_oktmo = dupes['oktmo_8'].iloc[0]
print(df_population_size[
    (df_population_size['oktmo_stable'] == problem_oktmo) &
    (df_population_size['mest'] == 'Все население') &
    (df_population_size['indicator_period'] == 'На 1 января') &
    (df_population_size['oktmo'] == df_population_size['oktmo_stable'])
][['municipality', 'mun_district', 'oktmo', 'oktmo_stable', 'year', 'indicator_value', 'mun_level']].to_string())

         municipality   mun_district     oktmo oktmo_stable  year  indicator_value                                  mun_level
207869  Новокузнецкий  Новокузнецкий  32731000     32731000  2009         563271.0  Муниципальное образование верхнего уровня
207872  Новокузнецкий  Новокузнецкий  32731000     32731000  2010         563507.0  Муниципальное образование верхнего уровня
207874  Новокузнецкий  Новокузнецкий  32731000     32731000  2011         548127.0  Муниципальное образование верхнего уровня
207876  Новокузнецкий  Новокузнецкий  32731000     32731000  2012         549589.0  Муниципальное образование верхнего уровня
207878  Новокузнецкий  Новокузнецкий  32731000     32731000  2013         549182.0  Муниципальное образование верхнего уровня
207879  Новокузнецкий  Новокузнецкий  32731000     32731000  2014         550213.0  Муниципальное образование верхнего уровня
207881  Новокузнецкий  Новокузнецкий  32731000     32731000  2015         550127.0  Муниципальное образование верхнего

In [26]:
print(df_population_size[
    (df_population_size['oktmo_stable'] == '79633000') &
    (df_population_size['mest'] == 'Все население') &
    (df_population_size['indicator_period'] == 'На 1 января') &
    (df_population_size['oktmo'] == df_population_size['oktmo_stable'])
][['municipality', 'mun_district', 'oktmo', 'oktmo_stable', 'year', 'indicator_value']].to_string())

                          municipality                    mun_district     oktmo oktmo_stable  year  indicator_value
444759  Теучежский муниципальный район  Теучежский муниципальный район  79633000     79633000  2009          20641.0
444762  Теучежский муниципальный район  Теучежский муниципальный район  79633000     79633000  2010          20584.0
444766  Теучежский муниципальный район  Теучежский муниципальный район  79633000     79633000  2011           5740.0
444767  Теучежский муниципальный район  Теучежский муниципальный район  79633000     79633000  2011          20651.0
444768  Теучежский муниципальный район  Теучежский муниципальный район  79633000     79633000  2012          20883.0
444772  Теучежский муниципальный район  Теучежский муниципальный район  79633000     79633000  2013          20887.0
444774  Теучежский муниципальный район  Теучежский муниципальный район  79633000     79633000  2014          21106.0
444779  Теучежский муниципальный район  Теучежский муниципальный

In [27]:
pop_clean = df_population_size[
    (df_population_size['mest'] == 'Все население') &
    (df_population_size['indicator_period'] == 'На 1 января') &
    (df_population_size['oktmo_stable'].str.len() == 8) &
    (df_population_size['oktmo'] == df_population_size['oktmo_stable'])
][['oktmo_stable', 'year', 'indicator_value']].rename(
    columns={'indicator_value': 'population', 'oktmo_stable': 'oktmo_8'}
)

pop_clean = (pop_clean
    .groupby(['oktmo_8', 'year'])['population']
    .max()
    .reset_index())

print(pop_clean.groupby(['oktmo_8', 'year']).size().max())

1


In [33]:
df_general = df_consumption[df_consumption['category'] == 'Все категории'][['date', 'territory_id',	'value']].copy()
df_general = df_general.merge(df_market_access, on='territory_id', how='left')
df_general = df_general.merge(
    df_dict_districts[df_dict_districts['year_to'] >= 2023][
        ['territory_id', 'oktmo_8', 'region_name', 'municipal_district_type']
    ].drop_duplicates('territory_id'),
    on='territory_id', how='left')
df_general = df_general.merge(df_key_rate_and_inflation[['date', 'key_rate', 'inflation']], on='date', how='left')
df_general = df_general.merge(
    df_consumer_spending[df_consumer_spending['type'] == 'Всего'][['date', 'national_spending']],
    on='date', how='left')
df_general = df_general.merge(df_currency_avg_monthly[['date', 'usd_rub']], on='date', how='left')
df_general['year'] = df_general['date'].dt.year
df_pop_clean = df_population_size[
    (df_population_size['mest'] == 'Все население') &
    (df_population_size['indicator_period'] == 'На 1 января') &
    (df_population_size['oktmo'].str.len() == 8)
][['oktmo', 'year', 'indicator_value']].rename(
    columns={'indicator_value': 'population', 'oktmo': 'oktmo_8'}
)
df_pop_clean = df_pop_clean.groupby(['oktmo_8', 'year'])['population'].max().reset_index()

df_head_clean = df_avg_org_headcount[
    (df_avg_org_headcount['okved2'] == 'Всего по обследуемым видам экономической деятельности') &
    (df_avg_org_headcount['indicator_period'] == 'Январь-декабрь') &
    (df_avg_org_headcount['mun_level'] == 'Муниципальное образование верхнего уровня')
][['oktmo', 'year', 'indicator_value']].rename(
    columns={'indicator_value': 'avg_headcount', 'oktmo': 'oktmo_8'}
)
df_head_clean = df_head_clean.groupby(['oktmo_8', 'year'])['avg_headcount'].max().reset_index()
df_general = df_general.merge(df_pop_clean, on=['oktmo_8', 'year'], how='left')
df_general = df_general.merge(df_head_clean, on=['oktmo_8', 'year'], how='left')

print(df_general.shape)
print(df_general.isnull().sum())

(50521, 14)
date                          0
territory_id                  0
value                         0
market_access               398
oktmo_8                       0
region_name                   0
municipal_district_type       0
key_rate                      0
inflation                     0
national_spending             0
usd_rub                       0
year                          0
population                 4283
avg_headcount              5244
dtype: int64


In [34]:
missing_mo = df_general[df_general['avg_headcount'].isna()]['oktmo_8'].unique()
print(f"Всего уникальных МО с пропуском: {len(missing_mo)}")
print(df_general[df_general['avg_headcount'].isna()].groupby('region_name')['oktmo_8'].nunique().sort_values(ascending=False))

Всего уникальных МО с пропуском: 328
region_name
Тверская область                       37
Новосибирская область                  35
Кемеровская область                    32
Рязанская область                      27
Нижегородская область                  24
Вологодская область                    22
Тамбовская область                     20
Чувашская Республика                   18
Курганская область                     16
Ставропольский край                    10
Приморский край                         9
Республика Северная Осетия — Алания     9
Магаданская область                     8
Забайкальский край                      8
Псковская область                       8
Архангельская область                   8
Амурская область                        7
Костромская область                     6
Новгородская область                    4
Республика Карелия                      4
Липецкая область                        4
Алтайский край                          3
Республика Коми            

In [35]:
df_general['market_access_imputed'] = df_general['market_access'].isna().astype(int)
df_general['population_imputed'] = df_general['population'].isna().astype(int)
df_general['headcount_imputed'] = df_general['avg_headcount'].isna().astype(int)

for col in ['market_access', 'population', 'avg_headcount']:
    df_general[col] = df_general.groupby('region_name')[col].transform(
        lambda x: x.fillna(x.median())
    )
print(df_general.isnull().sum())
print(df_general.shape)

date                       0
territory_id               0
value                      0
market_access              0
oktmo_8                    0
region_name                0
municipal_district_type    0
key_rate                   0
inflation                  0
national_spending          0
usd_rub                    0
year                       0
population                 0
avg_headcount              0
market_access_imputed      0
population_imputed         0
headcount_imputed          0
dtype: int64
(50521, 17)


In [36]:
total = df_consumption[df_consumption['category'] == 'Все категории'][
    ['date', 'territory_id', 'value']
].rename(columns={'value': 'total'})

df_cats = df_consumption[df_consumption['category'] != 'Все категории'].merge(
    total, on=['date', 'territory_id']
)
df_cats['cat_share'] = df_cats['value'] / df_cats['total']
df_cats_wide = df_cats.pivot_table(
    index=['date', 'territory_id'],
    columns='category',
    values='cat_share'
).reset_index()
df_cats_wide.columns = ['date', 'territory_id'] + [
    f'share_{c}' for c in df_cats_wide.columns[2:]
]
df_cats_wide.head()

,date,territory_id,share_Здоровье,share_Маркетплейсы,share_Общественное питание,share_Продовольствие,share_Транспорт
0,2023-01-01,1,0.058897,0.116080,0.052919,0.356441,0.079611
1,2023-01-01,2,0.070737,0.123354,0.065672,0.359876,0.090152
2,2023-01-01,3,0.053259,0.127361,0.022455,0.420486,0.058498
3,2023-01-01,4,0.066164,0.119408,0.024208,0.406701,0.079082
4,2023-01-01,5,0.061792,0.126121,0.019830,0.441572,0.078140


In [37]:
df_general = df_general.merge(df_cats_wide, on=['date', 'territory_id'], how='left')
df_general['month_sin'] = np.sin(2 * np.pi * df_general['date'].dt.month / 12)
df_general['month_cos'] = np.cos(2 * np.pi * df_general['date'].dt.month / 12)
ru_holidays = holidays.Russia()
def count_holidays(year, month):
    days = calendar.monthrange(year, month)[1]
    return sum(1 for d in range(1, days + 1)
              if pd.Timestamp(year, month, d) in ru_holidays)
df_general['holiday_count'] = df_general['date'].apply(
    lambda d: count_holidays(d.year, d.month)
)
df_general['spending_per_capita'] = df_general['value'] / df_general['population']
df_general = df_general.rename(columns={'value': 'total_spending'})
print(df_general.shape)
print(df_general.columns.tolist())

(50521, 26)
['date', 'territory_id', 'total_spending', 'market_access', 'oktmo_8', 'region_name', 'municipal_district_type', 'key_rate', 'inflation', 'national_spending', 'usd_rub', 'year', 'population', 'avg_headcount', 'market_access_imputed', 'population_imputed', 'headcount_imputed', 'share_Здоровье', 'share_Маркетплейсы', 'share_Общественное питание', 'share_Продовольствие', 'share_Транспорт', 'month_sin', 'month_cos', 'holiday_count', 'spending_per_capita']


In [38]:
df_general.isnull().sum()

date                          0
territory_id                  0
total_spending                0
market_access                 0
oktmo_8                       0
region_name                   0
municipal_district_type       0
key_rate                      0
inflation                     0
national_spending             0
usd_rub                       0
year                          0
population                    0
avg_headcount                 0
market_access_imputed         0
population_imputed            0
headcount_imputed             0
share_Здоровье                0
share_Маркетплейсы            0
share_Общественное питание    0
share_Продовольствие          0
share_Транспорт               0
month_sin                     0
month_cos                     0
holiday_count                 0
spending_per_capita           0
dtype: int64

In [39]:
df_general.to_parquet('../data/df_general.parquet', index=False)

print(f"Итоговый датафрейм: {df_general.shape}")
print(f"МО: {df_general['territory_id'].nunique()}")
print(f"Период: {df_general['date'].min()} — {df_general['date'].max()}")
print(f"Признаки: {df_general.columns.tolist()}")

Итоговый датафрейм: (50521, 26)
МО: 2190
Период: 2023-01-01 00:00:00 — 2024-12-01 00:00:00
Признаки: ['date', 'territory_id', 'total_spending', 'market_access', 'oktmo_8', 'region_name', 'municipal_district_type', 'key_rate', 'inflation', 'national_spending', 'usd_rub', 'year', 'population', 'avg_headcount', 'market_access_imputed', 'population_imputed', 'headcount_imputed', 'share_Здоровье', 'share_Маркетплейсы', 'share_Общественное питание', 'share_Продовольствие', 'share_Транспорт', 'month_sin', 'month_cos', 'holiday_count', 'spending_per_capita']
